# GROTH-05 : séries formelles multivariées en Mathlib — la marche manquante

**Navigation** : [<< Série SymbolicAI/Lean](../README.md) · [< GROTH-01 Catégories](GROTH-01-Categories-Mathlib-Lean.ipynb) · [> Lean-30 Groupes formels natifs](../Lean-30-FormalGroups-Native.ipynb) — kernel `lean4-wsl`, exécution native.

La carte cible de la série ([#17545](https://github.com/jsboige/CoursIA/issues/17545)) nommait cette marche comme allant de soi : « la marche "séries formelles multivariées" reste à écrire dans tous les cas ». [Lean-30](../Lean-30-FormalGroups-Native.ipynb) ouvre en effet `MvPowerSeries` à froid : l'étudiant y rencontre l'espace ambiant des groupes formels — `MvPowerSeries σ R` — sans qu'aucun carnet le lui ait fait manipuler coefficient par coefficient. Ce carnet est cette marche : après les catégories de [GROTH-01](GROTH-01-Categories-Mathlib-Lean.ipynb), il pose l'échafaudage algébrique que Lean-30 consomme.

Tout ce qui suit s'exécute nativement dans l'organe réel — `Mathlib.RingTheory.MvPowerSeries`, Mathlib `v4.33.0` (révision `db584cd6d46c`, la même que celle du lake `formal_groups_lean` de Lean-30) — sans réimplémentation jouet.

## Où ce carnet vous emmène

- **Lire** une série formelle multivariée pour ce qu'elle est : une fonction-coefficients sur les monômes `σ →₀ ℕ`, et la manipuler par ses coefficients.
- **Construire** avec les trois organes `C` (constantes), `X` (variables), `monomial` (monômes), et comprendre le produit de Cauchy coefficient par coefficient.
- **Cerner** l'idéal d'augmentation — les séries de terme constant nul — via `constantCoeff` : c'est la topologie qui donne son sens à un groupe formel.
- **Composer et inverser** : `subst` (substituer des séries dans une série — le geste même d'une loi de groupe formel) et `invOfUnit` (l'inverse d'une série à terme constant inversible, avec la série géométrique en témoin).

Prérequis : aucun carnet de la série n'est obligatoire ; GROTH-01 aide pour le vocabulaire d'organe (« où vit la sémantique »), mais les séries formelles y sont autonomes. Ce que ce carnet ne fait pas : la topologie `I`-adique complète et la théorie des groupes formels eux-mêmes — c'est le travail de Lean-30, qui étend ce vocabulaire par ses axiomes.

## 1. Où vit la sémantique — l'organe réel

La sémantique des séries formelles multivariées vit dans **Mathlib**, pas dans ce carnet : `Mathlib/RingTheory/MvPowerSeries/` — quatre fichiers que nous importons tous. `Basic.lean` porte le type et l'algèbre, `Substitution.lean` la composition, `Inverse.lean` l'inversion, `Order.lean` la graduation. Comme dans GROTH-01, chaque `#check` ci-dessous rend la **signature réelle** de l'organe : ce carnet n'affirme rien que le kernel n'ait imprimé.

In [1]:
import Mathlib.RingTheory.MvPowerSeries.Basic
import Mathlib.RingTheory.MvPowerSeries.Substitution
import Mathlib.RingTheory.MvPowerSeries.Order
import Mathlib.RingTheory.MvPowerSeries.Inverse

open MvPowerSeries

#check @MvPowerSeries
#check @MvPowerSeries.coeff
#check @MvPowerSeries.C
#check @MvPowerSeries.X
#check @MvPowerSeries.monomial
#check @MvPowerSeries.constantCoeff
#check @MvPowerSeries.subst
#check @MvPowerSeries.HasSubst
#check @MvPowerSeries.invOfUnit
#check @MvPowerSeries.order
#check @MvPowerSeries.map

import Mathlib.RingTheory.MvPowerSeries.Basic
import Mathlib.RingTheory.MvPowerSeries.Substitution
import Mathlib.RingTheory.MvPowerSeries.Order
import Mathlib.RingTheory.MvPowerSeries.Inverse

open MvPowerSeries

#check @MvPowerSeries
──────▶  MvPowerSeries : Type u_1 → Type u_2 → Type (max u_2 u_1)
#check @MvPowerSeries.coeff
──────▶  @coeff : {σ : Type u_1} → {R : Type u_2} → [inst : Semiring R] → (σ →₀ ℕ) → MvPowerSeries σ R →ₗ[R] R
#check @MvPowerSeries.C
──────▶  @C : {σ : Type u_1} → {R : Type u_2} → [inst : Semiring R] → R →+* MvPowerSeries σ R
#check @MvPowerSeries.X
──────▶  @X : {σ : Type u_1} → {R : Type u_2} → [Semiring R] → σ → MvPowerSeries σ R
#check @MvPowerSeries.monomial
──────▶  @monomial : {σ : Type u_1} → {R : Type u_2} → [inst : Semiring R] → (σ →₀ ℕ) → R →ₗ[R] MvPowerSeries σ R
#check @MvPowerSeries.constantCoeff
──────▶  @constantCoeff : {σ : Type u_1} → {R : Type u_2} → [inst : Semiring R] → MvPowerSeries σ R →+* R
#check @MvPowerSeries.subst
──────▶  @subst : {σ : Type u_1} →
  {R : Type u_2} →
    [inst : CommRing R] →
      {τ : Type u_3} →
        {S : Type u_4} →
          [inst_1 : CommRing S] → [Algebra R S] → (σ → MvPowerSeries τ S) → MvPowerSeries σ R → MvPowerSeries τ S
#check @MvPowerSeries.HasSubst
──────▶  @HasSubst : {σ : Type u_1} → {τ : Type u_2} → {S : Type u_3} → [CommRing S] → (σ → MvPowerSeries τ S) → Prop
#check @MvPowerSeries.invOfUnit
──────▶  @invOfUnit : {σ : Type u_1} → {R : Type u_2} → [inst : Ring R] → MvPowerSeries σ R → Rˣ → MvPowerSeries σ R
#check @MvPowerSeries.order
──────▶  @order : {σ : Type u_1} → {R : Type u_2} → [Semiring R] → MvPowerSeries σ R → ℕ∞
#check @MvPowerSeries.map
──────▶  @map : {σ : Type u_1} →
  {R : Type u_2} →
    {S : Type u_3} → [inst : Semiring R] → [inst_1 : Semiring S] → (R →+* S) → MvPowerSeries σ R →+* MvPowerSeries σ S
--% env 0

Raw input:
{"cmd": "import Mathlib.RingTheory.MvPowerSeries.Basic\nimport Mathlib.RingTheory.MvPowerSeries.Substitution\nimport Mathlib.RingTheory.MvPowerSeries.Order\nimport Mathlib.RingTheory.MvPowerSeries.Inverse\n\nopen MvPowerSeries\n\n#check @MvPowerSeries\n#check @MvPowerSeries.coeff\n#check @MvPowerSeries.C\n#check @MvPowerSeries.X\n#check @MvPowerSeries.monomial\n#check @MvPowerSeries.constantCoeff\n#check @MvPowerSeries.subst\n#check @MvPowerSeries.HasSubst\n#check @MvPowerSeries.invOfUnit\n#check @MvPowerSeries.order\n#check @MvPowerSeries.map"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data": "MvPowerSeries : Type u_1 → Type u_2 → Type (max u_2 u_1)"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "@coeff : {σ : Type u_1} → {R : Type u_2} → [inst : Semiring R] → (σ →₀ ℕ) → MvPowerSeries σ R →ₗ[R] R"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "@C : {σ : Type u_1} → {R : Type u_2} → [inst : Semiring R] → R →+* MvPowerSeries σ R"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "@X : {σ : Type u_1} → {R : Type u_2} → [Semiring R] → σ → MvPowerSeries σ R"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 6},
   "data":
   "@monomial : {σ : Type u_1} → {R : Type u_2} → [inst : Semiring R] → (σ →₀ ℕ) → R →ₗ[R] MvPowerSeries σ R"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 6},
   "data":
   "@constantCoeff : {σ : Type u_1} → {R : Type u_2} → [inst : Semiring R] → MvPowerSeries σ R →+* R"},
  {"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 6},
   "data":
   "@subst : {σ : Type u_1} →\n  {R : Type u_2} →\n    [inst : CommRing R] →\n      {τ : Type u_3} →\n        {S : Type u_4} →\n          [inst_1 : CommRing S] → [Algebra R S] → (σ → MvPowerSerie

### Lecture de la sortie

- `MvPowerSeries : Type u_1 → Type u_2 → Type (max u_2 u_1)` — le type prend **deux** paramètres : `σ` (l'ensemble des variables) puis `R` (l'anneau des coefficients). `MvPowerSeries (Fin 2) ℚ` est l'analogue formel de `ℚ[[x₀, x₁]]`.
- `coeff : (σ →₀ ℕ) → MvPowerSeries σ R →ₗ[R] R` — lire un coefficient. L'exposant `σ →₀ ℕ` est une fonction des variables vers `ℕ`, à support fini (`Finsupp`) : le monôme `x₀²x₁` y est la fonction valant 2 en `0`, 1 en `1`, 0 ailleurs. La flèche `→ₗ[R]` dit que `coeff m` est **linéaire en la série** — un morphisme de `R`-modules, pas une fonction nue.
- `C : R →+* MvPowerSeries σ R` et `constantCoeff : MvPowerSeries σ R →+* R` — deux morphismes d'anneaux (`→+*`), en sens inverse : plonger une constante, extraire le terme constant.
- `X : σ → MvPowerSeries σ R` — la famille des variables : `X i` est la série « la variable `i` ».
- `subst`, `HasSubst`, `invOfUnit`, `order`, `map` — les organes des sections 6 à 7 ; leurs signatures se liront au moment voulu.

## 2. Une série EST sa fonction-coefficients

Une série formelle n'est **pas** une expression : c'est la table complète de ses coefficients, indexée par les monômes. Pour un polynôme, la table est à support fini ; pour une série, elle est quelconque — c'est tout ce qui change. Concrètement : « connaître la série » et « savoir répondre `coeff m φ` pour tout monôme `m` » sont la même chose.

Notre témoin pour tout le carnet : `phi = 3 + x₀·x₁` sur `Fin 2` (deux variables) à coefficients dans `ℚ`. Le monôme `x₀x₁` s'écrit `Finsupp.single 0 1 + Finsupp.single 1 1` (1 en la variable 0, 1 en la variable 1, 0 ailleurs). Le monôme `1` est le `0` de `Finsupp` : la fonction constamment nulle.

In [2]:
/-- Série témoin : 3 + x0*x1, sur deux variables, à coefficients rationnels. -/
noncomputable def phi : MvPowerSeries (Fin 2) ℚ := C 3 + X 0 * X 1

-- Le terme constant : le coefficient du monôme nul.
example : (phi.coeff 0 : ℚ) = 3 := by simp [phi]

-- Le coefficient du monôme x0*x1 : un seul chemin de décomposition survit.
example : (phi.coeff (Finsupp.single 0 1 + Finsupp.single 1 1) : ℚ) = 1 := by
  simp only [phi, map_add]
  rw [coeff_C_of_ne_zero
      (by
        intro h
        have h0 : (Finsupp.single 0 1 + Finsupp.single 1 1 : Fin 2 →₀ ℕ) 0 = 1 := by simp
        rw [h] at h0
        simp at h0),
    zero_add, coeff_mul]
  rw [Finset.sum_eq_single_of_mem (Finsupp.single 0 1, Finsupp.single 1 1)
    (Finset.mem_antidiagonal.mpr rfl)
    (by
      intro p hp_mem hp_ne
      have hsum : p.1 + p.2 = Finsupp.single 0 1 + Finsupp.single 1 1 :=
        Finset.mem_antidiagonal.mp hp_mem
      rcases eq_or_ne p.1 (Finsupp.single 0 1) with h | h
      · exact absurd (Prod.ext h (by rw [h] at hsum; exact add_left_cancel hsum)) hp_ne
      · rw [coeff_X, if_neg h, zero_mul])]
  simp [coeff_index_single_self_X]

/-- Série témoin : 3 + x0*x1, sur deux variables, à coefficients rationnels. -/
noncomputable def phi : MvPowerSeries (Fin 2) ℚ := C 3 + X 0 * X 1

-- Le terme constant : le coefficient du monôme nul.
example : (phi.coeff 0 : ℚ) = 3 := by simp [phi]

-- Le coefficient du monôme x0*x1 : un seul chemin de décomposition survit.
example : (phi.coeff (Finsupp.single 0 1 + Finsupp.single 1 1) : ℚ) = 1 := by
  simp only [phi, map_add]
  rw [coeff_C_of_ne_zero
      (by
        intro h
        have h0 : (Finsupp.single 0 1 + Finsupp.single 1 1 : Fin 2 →₀ ℕ) 0 = 1 := by simp
        rw [h] at h0
        simp at h0),
    zero_add, coeff_mul]
  rw [Finset.sum_eq_single_of_mem (Finsupp.single 0 1, Finsupp.single 1 1)
    (Finset.mem_antidiagonal.mpr rfl)
    (by
      intro p hp_mem hp_ne
      have hsum : p.1 + p.2 = Finsupp.single 0 1 + Finsupp.single 1 1 :=
        Finset.mem_antidiagonal.mp hp_mem
      rcases eq_or_ne p.1 (Finsupp.single 0 1) with h | h
      · exact absurd (Prod.ext h (by rw [h] at hsum; exact add_left_cancel hsum)) hp_ne
      · rw [coeff_X, if_neg h, zero_mul])]
  simp [coeff_index_single_self_X]
--% env 1

Raw input:
{"cmd": "/-- S\u00e9rie t\u00e9moin : 3 + x0*x1, sur deux variables, \u00e0 coefficients rationnels. -/\nnoncomputable def phi : MvPowerSeries (Fin 2) \u211a := C 3 + X 0 * X 1\n\n-- Le terme constant : le coefficient du mon\u00f4me nul.\nexample : (phi.coeff 0 : \u211a) = 3 := by simp [phi]\n\n-- Le coefficient du mon\u00f4me x0*x1 : un seul chemin de d\u00e9composition survit.\nexample : (phi.coeff (Finsupp.single 0 1 + Finsupp.single 1 1) : \u211a) = 1 := by\n  simp only [phi, map_add]\n  rw [coeff_C_of_ne_zero\n      (by\n        intro h\n        have h0 : (Finsupp.single 0 1 + Finsupp.single 1 1 : Fin 2 \u2192\u2080 \u2115) 0 = 1 := by simp\n        rw [h] at h0\n        simp at h0),\n    zero_add, coeff_mul]\n  rw [Finset.sum_eq_single_of_mem (Finsupp.single 0 1, Finsupp.single 1 1)\n    (Finset.mem_antidiagonal.mpr rfl)\n    (by\n      intro p hp_mem hp_ne\n      have hsum : p.1 + p.2 = Finsupp.single 0 1 + Finsupp.single 1 1 :=\n        Finset.mem_antidiagonal.mp hp_mem\n      rcases eq_or_ne p.1 (Finsupp.single 0 1) with h | h\n      \u00b7 exact absurd (Prod.ext h (by rw [h] at hsum; exact add_left_cancel hsum)) hp_ne\n      \u00b7 rw [coeff_X, if_neg h, zero_mul])]\n  simp [coeff_index_single_self_X]", "env": 0}
Raw output:
{"env": 1}

### Lecture du résultat

La première égalité est close par `simp [phi]` ; la seconde est le **geste technique fondamental** du carnet : après déploiement de `phi` et du produit de Cauchy (`coeff_mul`), la somme porte sur les décompositions de l'exposant `x₀x₁`, et l'on prouve qu'**une seule survit** — `Finset.sum_eq_single_of_mem` exhibe le chemin `(x₀, x₁)` et annule tous les autres (chacun porte un facteur `coeff` nul). C'est la discipline de calcul de ce carnet : à ce pin Mathlib, les instances d'algèbre de `MvPowerSeries` sont **non calculables** (elles somment des familles infinies) — d'où le `noncomputable def` — et un coefficient se **lit par une preuve** (réécriture vers les lemmes `coeff_C`, `coeff_X`, `coeff_monomial`), jamais par un `#eval` sur la série elle-même. L'objet est défini ; ses valeurs se démontrent.

Retenez la lecture en grille : `phi` est la table `{monôme nul ↦ 3, x₀x₁ ↦ 1, tout le reste ↦ 0}`.

## 3. Les trois constructeurs : `C`, `X`, `monomial`

Toute série se décrit à partir de trois gestes : **constante** (`C a` — la série dont le seul coefficient non nul est le terme constant `a`), **variable** (`X i` — la série dont le coefficient de `xᵢ` vaut 1) et **monôme** (`monomial m a` — le monôme `m` affublé du coefficient `a`). Le premier est un morphisme d'anneaux, le troisième une application linéaire : Mathlib les typera toujours comme morphismes quand c'est vrai, et c'est une information, pas un ornement.

In [3]:
-- C : la constante, lue en deux coefficients.
example : ((C (3:ℚ) : MvPowerSeries (Fin 2) ℚ).coeff 0 : ℚ) = 3 := by simp
example : ((C (3:ℚ) : MvPowerSeries (Fin 2) ℚ).coeff (Finsupp.single 0 1) : ℚ) = 0 := by
  rw [coeff_C_of_ne_zero
    (by
      intro h
      have h0 : (Finsupp.single 0 1 : Fin 2 →₀ ℕ) 0 = 1 := by simp
      rw [h] at h0
      simp at h0)]

-- X : la variable, lue en deux coefficients.
example : ((X (0:Fin 2) : MvPowerSeries (Fin 2) ℚ).coeff (Finsupp.single 0 1) : ℚ) = 1 := by
  simp
example : ((X (0:Fin 2) : MvPowerSeries (Fin 2) ℚ).coeff (Finsupp.single 1 1) : ℚ) = 0 := by
  rw [coeff_index_single_X, if_neg (by decide)]

-- monomial : un monôme isolé, coefficient choisi.
example : ((monomial (Finsupp.single 1 2) (5:ℚ) : MvPowerSeries (Fin 2) ℚ)).coeff
    (Finsupp.single 1 2) = 5 := by simp
example : ((monomial (Finsupp.single 1 2) (5:ℚ) : MvPowerSeries (Fin 2) ℚ)).coeff
    (Finsupp.single 1 3) = 0 := by
  rw [coeff_monomial,
    if_neg
      (by
        intro h
        have h1 : (Finsupp.single 1 3 : Fin 2 →₀ ℕ) 1 = 3 := by simp
        rw [h] at h1
        simp at h1)]

-- C : la constante, lue en deux coefficients.
example : ((C (3:ℚ) : MvPowerSeries (Fin 2) ℚ).coeff 0 : ℚ) = 3 := by simp
example : ((C (3:ℚ) : MvPowerSeries (Fin 2) ℚ).coeff (Finsupp.single 0 1) : ℚ) = 0 := by
  rw [coeff_C_of_ne_zero
    (by
      intro h
      have h0 : (Finsupp.single 0 1 : Fin 2 →₀ ℕ) 0 = 1 := by simp
      rw [h] at h0
      simp at h0)]

-- X : la variable, lue en deux coefficients.
example : ((X (0:Fin 2) : MvPowerSeries (Fin 2) ℚ).coeff (Finsupp.single 0 1) : ℚ) = 1 := by
  simp
example : ((X (0:Fin 2) : MvPowerSeries (Fin 2) ℚ).coeff (Finsupp.single 1 1) : ℚ) = 0 := by
  rw [coeff_index_single_X, if_neg (by decide)]

-- monomial : un monôme isolé, coefficient choisi.
example : ((monomial (Finsupp.single 1 2) (5:ℚ) : MvPowerSeries (Fin 2) ℚ)).coeff
    (Finsupp.single 1 2) = 5 := by simp
example : ((monomial (Finsupp.single 1 2) (5:ℚ) : MvPowerSeries (Fin 2) ℚ)).coeff
    (Finsupp.single 1 3) = 0 := by
  rw [coeff_monomial,
    if_neg
      (by
        intro h
        have h1 : (Finsupp.single 1 3 : Fin 2 →₀ ℕ) 1 = 3 := by simp
        rw [h] at h1
        simp at h1)]
--% env 2

Raw input:
{"cmd": "-- C : la constante, lue en deux coefficients.\nexample : ((C (3:\u211a) : MvPowerSeries (Fin 2) \u211a).coeff 0 : \u211a) = 3 := by simp\nexample : ((C (3:\u211a) : MvPowerSeries (Fin 2) \u211a).coeff (Finsupp.single 0 1) : \u211a) = 0 := by\n  rw [coeff_C_of_ne_zero\n    (by\n      intro h\n      have h0 : (Finsupp.single 0 1 : Fin 2 \u2192\u2080 \u2115) 0 = 1 := by simp\n      rw [h] at h0\n      simp at h0)]\n\n-- X : la variable, lue en deux coefficients.\nexample : ((X (0:Fin 2) : MvPowerSeries (Fin 2) \u211a).coeff (Finsupp.single 0 1) : \u211a) = 1 := by\n  simp\nexample : ((X (0:Fin 2) : MvPowerSeries (Fin 2) \u211a).coeff (Finsupp.single 1 1) : \u211a) = 0 := by\n  rw [coeff_index_single_X, if_neg (by decide)]\n\n-- monomial : un mon\u00f4me isol\u00e9, coefficient choisi.\nexample : ((monomial (Finsupp.single 1 2) (5:\u211a) : MvPowerSeries (Fin 2) \u211a)).coeff\n    (Finsupp.single 1 2) = 5 := by simp\nexample : ((monomial (Finsupp.single 1 2) (5:\u211a) : MvPowerSeries (Fin 2) \u211a)).coeff\n    (Finsupp.single 1 3) = 0 := by\n  rw [coeff_monomial,\n    if_neg\n      (by\n        intro h\n        have h1 : (Finsupp.single 1 3 : Fin 2 \u2192\u2080 \u2115) 1 = 3 := by simp\n        rw [h] at h1\n        simp at h1)]", "env": 1}
Raw output:
{"env": 2}

### Lecture de la sortie

- Une constante **est** un monôme — celui d'exposant nul : `C 3 = monomial 0 3`. Les lectures ci-dessus le montrent coefficient par coefficient (3 au monôme nul, 0 ailleurs), qui est la seule égalité de séries qu'il y a à savoir : deux séries sont égales ssi tous leurs coefficients le sont (extensionnalité).
- Les deux lectures de `X 0` disent l'essentiel : coefficient 1 en son propre monôme, 0 en tout autre.
- `monomial m a` vaut `a` en `m` et `0` ailleurs : c'est la « cellule de grille » dont une série est la somme (infinie).

## 4. Le produit de Cauchy — l'algèbre des séries

Multiplier deux séries, c'est convoluer leurs grilles : le coefficient du monôme `m` dans `f * g` est la somme des `f p * g q` sur toutes les façons de **décomposer `m` en `p + q`**. Cette somme porte sur un ensemble fini (les décompositions d'un exposant fixé) : c'est ce qui rend le produit bien défini sans aucune convergence à invoquer. L'organe s'appelle `coeff_mul`.

In [4]:
#check @MvPowerSeries.coeff_mul

-- Arithmétique des exposants : x0^2 est la somme des deux chemins (x0, x0).
example : Finsupp.single 0 1 + Finsupp.single 0 1 = Finsupp.single 0 2 := by
  rw [← Finsupp.single_add]

-- x0 * x1 : vers le monôme x0x1, le seul chemin de décomposition est (x0, x1).
example : ((X 0 * X 1 : MvPowerSeries (Fin 2) ℚ).coeff
    (Finsupp.single 0 1 + Finsupp.single 1 1) : ℚ) = 1 := by
  rw [coeff_mul]
  rw [Finset.sum_eq_single_of_mem (Finsupp.single 0 1, Finsupp.single 1 1)
    (Finset.mem_antidiagonal.mpr rfl)
    (by
      intro p hp_mem hp_ne
      have hsum : p.1 + p.2 = Finsupp.single 0 1 + Finsupp.single 1 1 :=
        Finset.mem_antidiagonal.mp hp_mem
      rcases eq_or_ne p.1 (Finsupp.single 0 1) with h | h
      · exact absurd (Prod.ext h (by rw [h] at hsum; exact add_left_cancel hsum)) hp_ne
      · rw [coeff_X, if_neg h, zero_mul])]
  simp [coeff_index_single_self_X]

-- x0 * x0 : coefficient 1 en x0^2 — même discipline, un seul chemin non nul.
example : ((X 0 * X 0 : MvPowerSeries (Fin 2) ℚ).coeff (Finsupp.single 0 2) : ℚ) = 1 := by
  rw [coeff_mul]
  rw [Finset.sum_eq_single_of_mem (Finsupp.single 0 1, Finsupp.single 0 1)
    (Finset.mem_antidiagonal.mpr
      (by
        show Finsupp.single 0 1 + Finsupp.single 0 1 = Finsupp.single 0 2
        rw [← Finsupp.single_add]))
    (by
      intro p hp_mem hp_ne
      have hsum : p.1 + p.2 = Finsupp.single 0 1 + Finsupp.single 0 1 := by
        rw [Finset.mem_antidiagonal.mp hp_mem, ← Finsupp.single_add]
      rcases eq_or_ne p.1 (Finsupp.single 0 1) with h | h
      · exact absurd (Prod.ext h (by rw [h] at hsum; exact add_left_cancel hsum)) hp_ne
      · rw [coeff_X, if_neg h, zero_mul])]
  simp [coeff_index_single_self_X]

-- Les constantes se multiplient terme à terme : C 2 * C 5 = C 10.
example : ((C 2 * C 5 : MvPowerSeries (Fin 2) ℚ).coeff 0 : ℚ) = 10 := by
  simp <;> norm_num

#check @MvPowerSeries.coeff_mul
──────▶  @coeff_mul : ∀ {σ : Type u_1} {R : Type u_2} [inst : Semiring R] (n : σ →₀ ℕ) (φ ψ : MvPowerSeries σ R)
  [inst_1 : DecidableEq σ], (coeff n) (φ * ψ) = ∑ p ∈ Finset.antidiagonal n, (coeff p.1) φ * (coeff p.2) ψ

-- Arithmétique des exposants : x0^2 est la somme des deux chemins (x0, x0).
example : Finsupp.single 0 1 + Finsupp.single 0 1 = Finsupp.single 0 2 := by
  rw [← Finsupp.single_add]

-- x0 * x1 : vers le monôme x0x1, le seul chemin de décomposition est (x0, x1).
example : ((X 0 * X 1 : MvPowerSeries (Fin 2) ℚ).coeff
    (Finsupp.single 0 1 + Finsupp.single 1 1) : ℚ) = 1 := by
  rw [coeff_mul]
  rw [Finset.sum_eq_single_of_mem (Finsupp.single 0 1, Finsupp.single 1 1)
    (Finset.mem_antidiagonal.mpr rfl)
    (by
      intro p hp_mem hp_ne
      have hsum : p.1 + p.2 = Finsupp.single 0 1 + Finsupp.single 1 1 :=
        Finset.mem_antidiagonal.mp hp_mem
      rcases eq_or_ne p.1 (Finsupp.single 0 1) with h | h
      · exact absurd (Prod.ext h (by rw [h] at hsum; exact add_left_cancel hsum)) hp_ne
      · rw [coeff_X, if_neg h, zero_mul])]
  simp [coeff_index_single_self_X]

-- x0 * x0 : coefficient 1 en x0^2 — même discipline, un seul chemin non nul.
example : ((X 0 * X 0 : MvPowerSeries (Fin 2) ℚ).coeff (Finsupp.single 0 2) : ℚ) = 1 := by
  rw [coeff_mul]
  rw [Finset.sum_eq_single_of_mem (Finsupp.single 0 1, Finsupp.single 0 1)
    (Finset.mem_antidiagonal.mpr
      (by
        show Finsupp.single 0 1 + Finsupp.single 0 1 = Finsupp.single 0 2
        rw [← Finsupp.single_add]))
    (by
      intro p hp_mem hp_ne
      have hsum : p.1 + p.2 = Finsupp.single 0 1 + Finsupp.single 0 1 := by
        rw [Finset.mem_antidiagonal.mp hp_mem, ← Finsupp.single_add]
      rcases eq_or_ne p.1 (Finsupp.single 0 1) with h | h
      · exact absurd (Prod.ext h (by rw [h] at hsum; exact add_left_cancel hsum)) hp_ne
      · rw [coeff_X, if_neg h, zero_mul])]
  simp [coeff_index_single_self_X]

-- Les constantes se multiplient terme à terme : C 2 * C 5 = C 10.
example : ((C 2 * C 5 : MvPowerSeries (Fin 2) ℚ).coeff 0 : ℚ) = 10 := by
  simp <;> norm_num
       ───▶ 🟨 Used `tac1 <;> tac2` where `(tac1; tac2)` would suffice

Note: This linter can be disabled with `set_option linter.unnecessarySeqFocus false`
--% env 3

Raw input:
{"cmd": "#check @MvPowerSeries.coeff_mul\n\n-- Arithm\u00e9tique des exposants : x0^2 est la somme des deux chemins (x0, x0).\nexample : Finsupp.single 0 1 + Finsupp.single 0 1 = Finsupp.single 0 2 := by\n  rw [\u2190 Finsupp.single_add]\n\n-- x0 * x1 : vers le mon\u00f4me x0x1, le seul chemin de d\u00e9composition est (x0, x1).\nexample : ((X 0 * X 1 : MvPowerSeries (Fin 2) \u211a).coeff\n    (Finsupp.single 0 1 + Finsupp.single 1 1) : \u211a) = 1 := by\n  rw [coeff_mul]\n  rw [Finset.sum_eq_single_of_mem (Finsupp.single 0 1, Finsupp.single 1 1)\n    (Finset.mem_antidiagonal.mpr rfl)\n    (by\n      intro p hp_mem hp_ne\n      have hsum : p.1 + p.2 = Finsupp.single 0 1 + Finsupp.single 1 1 :=\n        Finset.mem_antidiagonal.mp hp_mem\n      rcases eq_or_ne p.1 (Finsupp.single 0 1) with h | h\n      \u00b7 exact absurd (Prod.ext h (by rw [h] at hsum; exact add_left_cancel hsum)) hp_ne\n      \u00b7 rw [coeff_X, if_neg h, zero_mul])]\n  simp [coeff_index_single_self_X]\n\n-- x0 * x0 : coefficient 1 en x0^2 \u2014 m\u00eame discipline, un seul chemin non nul.\nexample : ((X 0 * X 0 : MvPowerSeries (Fin 2) \u211a).coeff (Finsupp.single 0 2) : \u211a) = 1 := by\n  rw [coeff_mul]\n  rw [Finset.sum_eq_single_of_mem (Finsupp.single 0 1, Finsupp.single 0 1)\n    (Finset.mem_antidiagonal.mpr\n      (by\n        show Finsupp.single 0 1 + Finsupp.single 0 1 = Finsupp.single 0 2\n        rw [\u2190 Finsupp.single_add]))\n    (by\n      intro p hp_mem hp_ne\n      have hsum : p.1 + p.2 = Finsupp.single 0 1 + Finsupp.single 0 1 := by\n        rw [Finset.mem_antidiagonal.mp hp_mem, \u2190 Finsupp.single_add]\n      rcases eq_or_ne p.1 (Finsupp.single 0 1) with h | h\n      \u00b7 exact absur

### Interprétation

La signature de `coeff_mul` dit la convolution en formule : `(f * g).coeff m = ∑ sur les décompositions p + q = m`. Nos vérifications la rejouent avec l'organe `Finset.sum_eq_single_of_mem` : vers `x₀x₁`, tout chemin autre que `(x₀, x₁)` porte un facteur nul — sinon `p₁ = x₀` et la décomposition force `p₂ = x₁`, contradiction. Idem vers `x₀²`, en passant par l'arithmétique des exposants (`single 0 1 + single 0 1 = single 0 2`). Et le produit des constantes redescend au produit de l'anneau. Tout ce que Lean-30 fera — composer des lois, itérer — est déjà cette convolution, emballée.

Si vous avez en tête `ℚ[[x]]` à une variable : rien ne change, sauf que les exposants deviennent des fonctions `σ →₀ ℕ` et que les décompositions `p + q` se comptent sur cette grille-là.

## 5. Terme constant et idéal d'augmentation

Le morphisme `constantCoeff : MvPowerSeries σ R →+* R` extrait le coefficient du monôme nul. Son **noyau** — les séries de terme constant nul — est l'**idéal d'augmentation** `I` : c'est lui qui donne aux séries formelles leur topologie (dire que des séries sont « de plus en plus proches », c'est dire que leur différence vit dans `I`, puis `I²`, puis `I³`…). L'ordre `order` (signature imprimée en section 1) mesure le plus petit degré total auquel la série a un coefficient non nul — `∞` pour la série nulle.

In [5]:
example : (phi.constantCoeff : ℚ) = 3 := by simp [phi]
example : ((X 0 * X 1 : MvPowerSeries (Fin 2) ℚ).constantCoeff : ℚ) = 0 := by simp

#check @MvPowerSeries.order
#check @MvPowerSeries.coeff_mul_left_one_sub_of_lt_order

example : (phi.constantCoeff : ℚ) = 3 := by simp [phi]
example : ((X 0 * X 1 : MvPowerSeries (Fin 2) ℚ).constantCoeff : ℚ) = 0 := by simp

#check @MvPowerSeries.order
──────▶  @order : {σ : Type u_1} → {R : Type u_2} → [Semiring R] → MvPowerSeries σ R → ℕ∞
#check @MvPowerSeries.coeff_mul_left_one_sub_of_lt_order
──────▶  @coeff_mul_left_one_sub_of_lt_order : ∀ {σ : Type u_1} {R : Type u_2} [inst : Ring R] {f g : MvPowerSeries σ R}
  (d : σ →₀ ℕ), ↑(Finsupp.degree d) < g.order → (coeff d) (f * (1 - g)) = (coeff d) f
--% env 4

Raw input:
{"cmd": "example : (phi.constantCoeff : \u211a) = 3 := by simp [phi]\nexample : ((X 0 * X 1 : MvPowerSeries (Fin 2) \u211a).constantCoeff : \u211a) = 0 := by simp\n\n#check @MvPowerSeries.order\n#check @MvPowerSeries.coeff_mul_left_one_sub_of_lt_order", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@order : {σ : Type u_1} → {R : Type u_2} → [Semiring R] → MvPowerSeries σ R → ℕ∞"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "@coeff_mul_left_one_sub_of_lt_order : ∀ {σ : Type u_1} {R : Type u_2} [inst : Ring R] {f g : MvPowerSeries σ R}\n  (d : σ →₀ ℕ), ↑(Finsupp.degree d) < g.order → (coeff d) (f * (1 - g)) = (coeff d) f"}],
 "env": 4}

### Lecture du résultat

`constantCoeff` est un morphisme d'anneaux : il respecte somme et produit — c'est ce qui fait de son noyau un **idéal**. Le lemme `coeff_mul_left_one_sub_of_lt_order` imprimé ci-dessus est la brique « topologie » : multiplier par `(1 - quelque chose de degré ≥ 1)` ne laisse rien sous un certain ordre — exactement l'intuition `I · I^n ⊆ I^(n+1)` qui fait converger les sommes infinies de la section 7.

Pour un groupe formel, tout se passera **dans** l'idéal d'augmentation : une loi de groupe formel vit sur les séries de terme constant nul, et l'axiome d'identité dira que substituer zéro aux variables rend zéro.

## 6. La substitution — composer les séries

Voici le geste central pour la suite de la série : **substituer une famille de séries aux variables d'une série**. La signature lue en section 1 — `subst : (σ → MvPowerSeries τ S) → MvPowerSeries σ R → MvPowerSeries τ S` — prend une famille `g : σ → séries` (une série cible par variable) et une série `φ` en les variables `σ`, et rend `φ` réécrite en remplaçant chaque `xᵢ` par `g i`. La condition `HasSubst g` — chaque `g i` de terme constant nul — garantit que chaque coefficient du résultat est une somme **finie** : pour calculer le coefficient de degré `d`, seuls les monômes de `φ` de degré ≤ `d` contribuent.

In [6]:
/-- Une famille substituante : chaque variable part vers une série de terme constant nul. -/
noncomputable def fam : Fin 2 → MvPowerSeries (Fin 2) ℚ :=
  fun i => X i + X 0 * X 1

#check (MvPowerSeries.subst fam phi : MvPowerSeries (Fin 2) ℚ)

-- Substituer la famille nulle : il ne reste que le terme constant, via l'organe.
example : MvPowerSeries.subst (0 : Fin 2 → MvPowerSeries (Fin 2) ℚ) phi = C 3 := by
  rw [MvPowerSeries.subst_zero_eq_C_constantCoeff]
  simp [phi]

/-- Une famille substituante : chaque variable part vers une série de terme constant nul. -/
noncomputable def fam : Fin 2 → MvPowerSeries (Fin 2) ℚ :=
  fun i => X i + X 0 * X 1

#check (MvPowerSeries.subst fam phi : MvPowerSeries (Fin 2) ℚ)
──────▶  subst fam phi : MvPowerSeries (Fin 2) ℚ

-- Substituer la famille nulle : il ne reste que le terme constant, via l'organe.
example : MvPowerSeries.subst (0 : Fin 2 → MvPowerSeries (Fin 2) ℚ) phi = C 3 := by
  rw [MvPowerSeries.subst_zero_eq_C_constantCoeff]
  simp [phi]
--% env 5

Raw input:
{"cmd": "/-- Une famille substituante : chaque variable part vers une s\u00e9rie de terme constant nul. -/\nnoncomputable def fam : Fin 2 \u2192 MvPowerSeries (Fin 2) \u211a :=\n  fun i => X i + X 0 * X 1\n\n#check (MvPowerSeries.subst fam phi : MvPowerSeries (Fin 2) \u211a)\n\n-- Substituer la famille nulle : il ne reste que le terme constant, via l'organe.\nexample : MvPowerSeries.subst (0 : Fin 2 \u2192 MvPowerSeries (Fin 2) \u211a) phi = C 3 := by\n  rw [MvPowerSeries.subst_zero_eq_C_constantCoeff]\n  simp [phi]", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data": "subst fam phi : MvPowerSeries (Fin 2) ℚ"}],
 "env": 5}

### Interprétation

La dernière égalité n'est pas anecdotique : `subst_zero_eq_C_constantCoeff` (**lemme de l'organe**, `Substitution.lean`) dit que substituer la famille nulle à toute série la réduit à sa constante — pour `phi`, `C 3`. En y pensant comme « évaluer la série en `(0, …, 0)` », c'est **l'axiome d'identité d'un groupe formel** déjà présent, tout nu : la loi évaluée au point nul rend le point nul dès que son terme constant l'est (section 8).

Pourquoi `HasSubst` exige le terme constant nul : substituer `g` à `x` dans `Σ xⁿ` produirait `Σ (g)ⁿ`, une somme infinie **par coefficient** — nulle part finie si `g` a un terme constant non nul. La condition ramène chaque coefficient à une somme finie, sans quoi `subst` ne serait pas définissable.

## 7. Inverser une série : `invOfUnit` et la série géométrique

Qui peut être inversé pour la multiplication ? Réponse de l'organe (`Inverse.lean`) : **une série est une unité si et seulement si son terme constant l'est** — `isUnit_iff_constantCoeff`. C'est la marque du caractère local de l'anneau des séries. L'inverse explicite se construit par `invOfUnit` : la série géométrique en est le témoin canonique, `1/(1+x₀) = 1 - x₀ + x₀² - x₀³ + …`

In [7]:
#check @MvPowerSeries.isUnit_iff_constantCoeff
#check @MvPowerSeries.invOfUnit
#check @MvPowerSeries.invOfUnit_mul
#check @MvPowerSeries.constantCoeff_invOfUnit

/-- La série géométrique : inverse de 1 + x0 — l'unité 1 de ℚˣ en témoin. -/
noncomputable def geo : MvPowerSeries (Fin 2) ℚ :=
  MvPowerSeries.invOfUnit (C 1 + X 0) 1

-- La propriété définissante : geo * (1 + x0) = 1.
example : (geo * (C 1 + X 0) : MvPowerSeries (Fin 2) ℚ) = 1 := by
  rw [geo]
  exact MvPowerSeries.invOfUnit_mul _ 1 (by simp)

-- Son terme constant : l'inverse de l'unité, lu par l'organe.
example : (geo.constantCoeff : ℚ) = 1 := by
  rw [geo, MvPowerSeries.constantCoeff_invOfUnit]
  simp

#check @MvPowerSeries.isUnit_iff_constantCoeff
──────▶  @isUnit_iff_constantCoeff : ∀ {σ : Type u_1} {R : Type u_2} [inst : Ring R] {φ : MvPowerSeries σ R},
  IsUnit φ ↔ IsUnit (constantCoeff φ)
#check @MvPowerSeries.invOfUnit
──────▶  @invOfUnit : {σ : Type u_1} → {R : Type u_2} → [inst : Ring R] → MvPowerSeries σ R → Rˣ → MvPowerSeries σ R
#check @MvPowerSeries.invOfUnit_mul
──────▶  @invOfUnit_mul : ∀ {σ : Type u_1} {R : Type u_2} [inst : Ring R] (φ : MvPowerSeries σ R) (u : Rˣ),
  constantCoeff φ = ↑u → φ.invOfUnit u * φ = 1
#check @MvPowerSeries.constantCoeff_invOfUnit
──────▶  @constantCoeff_invOfUnit : ∀ {σ : Type u_1} {R : Type u_2} [inst : Ring R] (φ : MvPowerSeries σ R) (u : Rˣ),
  constantCoeff (φ.invOfUnit u) = ↑u⁻¹

/-- La série géométrique : inverse de 1 + x0 — l'unité 1 de ℚˣ en témoin. -/
noncomputable def geo : MvPowerSeries (Fin 2) ℚ :=
  MvPowerSeries.invOfUnit (C 1 + X 0) 1

-- La propriété définissante : geo * (1 + x0) = 1.
example : (geo * (C 1 + X 0) : MvPowerSeries (Fin 2) ℚ) = 1 := by
  rw [geo]
  exact MvPowerSeries.invOfUnit_mul _ 1 (by simp)

-- Son terme constant : l'inverse de l'unité, lu par l'organe.
example : (geo.constantCoeff : ℚ) = 1 := by
  rw [geo, MvPowerSeries.constantCoeff_invOfUnit]
  simp
--% env 6

Raw input:
{"cmd": "#check @MvPowerSeries.isUnit_iff_constantCoeff\n#check @MvPowerSeries.invOfUnit\n#check @MvPowerSeries.invOfUnit_mul\n#check @MvPowerSeries.constantCoeff_invOfUnit\n\n/-- La s\u00e9rie g\u00e9om\u00e9trique : inverse de 1 + x0 \u2014 l'unit\u00e9 1 de \u211a\u02e3 en t\u00e9moin. -/\nnoncomputable def geo : MvPowerSeries (Fin 2) \u211a :=\n  MvPowerSeries.invOfUnit (C 1 + X 0) 1\n\n-- La propri\u00e9t\u00e9 d\u00e9finissante : geo * (1 + x0) = 1.\nexample : (geo * (C 1 + X 0) : MvPowerSeries (Fin 2) \u211a) = 1 := by\n  rw [geo]\n  exact MvPowerSeries.invOfUnit_mul _ 1 (by simp)\n\n-- Son terme constant : l'inverse de l'unit\u00e9, lu par l'organe.\nexample : (geo.constantCoeff : \u211a) = 1 := by\n  rw [geo, MvPowerSeries.constantCoeff_invOfUnit]\n  simp", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "@isUnit_iff_constantCoeff : ∀ {σ : Type u_1} {R : Type u_2} [inst : Ring R] {φ : MvPowerSeries σ R},\n  IsUnit φ ↔ IsUnit (constantCoeff φ)"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@invOfUnit : {σ : Type u_1} → {R : Type u_2} → [inst : Ring R] → MvPowerSeries σ R → Rˣ → MvPowerSeries σ R"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@invOfUnit_mul : ∀ {σ : Type u_1} {R : Type u_2} [inst : Ring R] (φ : MvPowerSeries σ R) (u : Rˣ),\n  constantCoeff φ = ↑u → φ.invOfUnit u * φ = 1"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@constantCoeff_invOfUnit : ∀ {σ : Type u_1} {R : Type u_2} [inst : Ring R] (φ : MvPowerSeries σ R) (u : Rˣ),\n  constantCoeff (φ.invOfUnit u) = ↑u⁻¹"}],
 "env": 6}

### Lecture du résultat

`invOfUnit (C 1 + X 0) 1` prend **l'unité `u : Rˣ`** dont le terme constant est la valeur — ici `1 : ℚˣ` — et `invOfUnit_mul` exige la preuve `constantCoeff φ = ↑u` (fermée par `simp`). L'égalité `geo · (1 + x₀) = 1` **est** la somme géométrique infinie `geo = 1 - x₀ + x₀² - …`, vue de l'unique façon honnête en formel : par la propriété qui la caractérise. Le kernel vérifie l'identité tout entière — chaque coefficient de la convolution tombe au bon endroit. C'est le même organe qui, dans un groupe formel, fournira l'inverse d'un élément : l'axiome d'inverse d'une loi `F` revient à inverser une série linéaire en `X` à terme constant 1.

## 8. Le pont vers les groupes formels

Tout est maintenant en place pour lire [Lean-30](../Lean-30-FormalGroups-Native.ipynb). Une **loi de groupe formel commutatif de dimension `g`** sur `R`, c'est une série `F` en `2g` variables — `MvPowerSeries (Fin g ⊕ Fin g) R` — de terme constant nul, linéaire en chaque bloc, et telle que : substituer les deux blocs d'identité à `F` redonne le bloc (`identité`), l'inverse existe (`inverse`, la section 7), et `F(F(x,y),z) = F(x,F(y,z))` (`associativité` — une égalité de substitutions). Le lake `formal_groups_lean` de Lean-30 définit le record `MvFormalGroup` avec ces axiomes ; ce carnet a construit chaque pièce du vocabulaire qu'ils emploient.

La loi additive, lue dans l'organe, et son axiome d'identité démontrés par les pièces des sections précédentes :

In [8]:
/-- La loi additive F(x, y) = x + y sur deux blocs d'une variable. -/
noncomputable def addLaw : MvPowerSeries (Fin 1 ⊕ Fin 1) ℚ :=
  X (Sum.inl 0) + X (Sum.inr 0)

-- F est de terme constant nul : elle vit dans l'idéal d'augmentation.
example : (addLaw.constantCoeff : ℚ) = 0 := by simp [addLaw]

-- Axiome d'identité : substituer la famille nulle à F rend la série nulle.
example : MvPowerSeries.subst
    (0 : (Fin 1 ⊕ Fin 1) → MvPowerSeries (Fin 1 ⊕ Fin 1) ℚ) addLaw = C 0 := by
  rw [MvPowerSeries.subst_zero_eq_C_constantCoeff]
  simp [addLaw]

/-- La loi additive F(x, y) = x + y sur deux blocs d'une variable. -/
noncomputable def addLaw : MvPowerSeries (Fin 1 ⊕ Fin 1) ℚ :=
  X (Sum.inl 0) + X (Sum.inr 0)

-- F est de terme constant nul : elle vit dans l'idéal d'augmentation.
example : (addLaw.constantCoeff : ℚ) = 0 := by simp [addLaw]

-- Axiome d'identité : substituer la famille nulle à F rend la série nulle.
example : MvPowerSeries.subst
    (0 : (Fin 1 ⊕ Fin 1) → MvPowerSeries (Fin 1 ⊕ Fin 1) ℚ) addLaw = C 0 := by
  rw [MvPowerSeries.subst_zero_eq_C_constantCoeff]
  simp [addLaw]
--% env 7

Raw input:
{"cmd": "/-- La loi additive F(x, y) = x + y sur deux blocs d'une variable. -/\nnoncomputable def addLaw : MvPowerSeries (Fin 1 \u2295 Fin 1) \u211a :=\n  X (Sum.inl 0) + X (Sum.inr 0)\n\n-- F est de terme constant nul : elle vit dans l'id\u00e9al d'augmentation.\nexample : (addLaw.constantCoeff : \u211a) = 0 := by simp [addLaw]\n\n-- Axiome d'identit\u00e9 : substituer la famille nulle \u00e0 F rend la s\u00e9rie nulle.\nexample : MvPowerSeries.subst\n    (0 : (Fin 1 \u2295 Fin 1) \u2192 MvPowerSeries (Fin 1 \u2295 Fin 1) \u211a) addLaw = C 0 := by\n  rw [MvPowerSeries.subst_zero_eq_C_constantCoeff]\n  simp [addLaw]", "env": 6}
Raw output:
{"env": 7}

### Interprétation

`subst_zero_eq_C_constantCoeff` referme l'axiome en une réécriture : évaluée au point nul, la loi `x + y` rend `C 0`, la série nulle. Ce que Lean-30 ajoute — et qui reste son travail : le record `MvFormalGroup`, la preuve que `addMv` en est un exemple (vous y croiserez `(addMv 1 ℤ)` et ses lemmes `constantCoeff_eq_zero`, `coeff_single_inl`), les morphismes, les itérés et la hauteur. Ce carnet referme la marche que la carte disait « à écrire dans tous les cas » : le vocabulaire est posé, l'organe a été exécuté.

## 9. Intégrité : sur quoi repose ce carnet

Comme pour GROTH-01, la transparence axiomatique se lit au kernel : `#print axioms` rend la liste complète des axiomes dont chaque définition dépend. Les trois attendus — `propext` (extensionnalité des propositions), `Quot.sound` (les quotients), `Classical.choice` (le choix classique, ici entraîné par les instances non calculables) — et **pas de `sorryAx`** : rien dans ce carnet ne repose sur une preuve admise.

In [9]:
#print axioms phi
#print axioms fam
#print axioms geo
#print axioms addLaw

#print axioms phi
──────▶  'phi' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms fam
──────▶  'fam' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms geo
──────▶  'geo' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms addLaw
──────▶  'addLaw' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 8

Raw input:
{"cmd": "#print axioms phi\n#print axioms fam\n#print axioms geo\n#print axioms addLaw", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "'phi' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "'fam' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "'geo' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "'addLaw' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 8}

## 10. Exercices

Trois exercices pour s'approprier les organes, du plus mécanique au plus conceptuel. Aucun ne reprend les exemples du corps du carnet.

### Exercice 1 : lire une série

**Objectif** : définir `psi = C 5 + X 1 * X 1` et démontrer ses deux premières lectures — le terme constant vaut 5, le coefficient de `x₁²` vaut 1.

In [10]:
-- Exercice 1 : la serie psi et ses deux premieres lectures
--
-- TODO etudiant :
-- noncomputable def psi : MvPowerSeries (Fin 2) ℚ := C 5 + X 1 * X 1
-- example : (psi.coeff 0 : ℚ) = 5 := by simp [psi]
-- example : (psi.coeff (Finsupp.single 1 2) : ℚ) = 1 := by simp [psi]
--
-- (indice : recopier la structure de la section 2 en changeant la serie ;
--  le monome x1^2 s'ecrit Finsupp.single 1 2.)

#eval "Exercice a completer"

-- Exercice 1 : la serie psi et ses deux premieres lectures
--
-- TODO etudiant :
-- noncomputable def psi : MvPowerSeries (Fin 2) ℚ := C 5 + X 1 * X 1
-- example : (psi.coeff 0 : ℚ) = 5 := by simp [psi]
-- example : (psi.coeff (Finsupp.single 1 2) : ℚ) = 1 := by simp [psi]
--
-- (indice : recopier la structure de la section 2 en changeant la serie ;
--  le monome x1^2 s'ecrit Finsupp.single 1 2.)

#eval "Exercice a completer"
─────▶  "Exercice a completer"
--% env 9

Raw input:
{"cmd": "-- Exercice 1 : la serie psi et ses deux premieres lectures\n--\n-- TODO etudiant :\n-- noncomputable def psi : MvPowerSeries (Fin 2) \u211a := C 5 + X 1 * X 1\n-- example : (psi.coeff 0 : \u211a) = 5 := by simp [psi]\n-- example : (psi.coeff (Finsupp.single 1 2) : \u211a) = 1 := by simp [psi]\n--\n-- (indice : recopier la structure de la section 2 en changeant la serie ;\n--  le monome x1^2 s'ecrit Finsupp.single 1 2.)\n\n#eval \"Exercice a completer\"", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "\"Exercice a completer\""}],
 "env": 9}

### Exercice 2 : composer par substitution

**Objectif** : définir la famille `swap` qui échange les deux variables (`x₀ ↦ x₁`, `x₁ ↦ x₀`) et montrer que la substitution préserve le terme constant de `phi` : `constantCoeff (subst swap phi) = 3`. C'est le cœur du geste « évaluer une loi en un point de l'idéal ».

In [11]:
-- Exercice 2 : l'echange des variables preserve le terme constant
--
-- TODO etudiant :
-- noncomputable def swap : Fin 2 → MvPowerSeries (Fin 2) ℚ :=
--   fun i => X (if i = 0 then 1 else 0)
-- example : (MvPowerSeries.subst swap phi).constantCoeff = 3 := by
--   ...
--
-- (indice : chaque X j est de terme constant nul, donc la substitution ne peut
--  pas creer de terme constant : seul C 3 de phi y contribue. Commencer par
--  montrer constantCoeff (swap i) = 0 pour tout i, puis conclure comme en
--  section 6.)

#eval "Exercice a completer"

-- Exercice 2 : l'echange des variables preserve le terme constant
--
-- TODO etudiant :
-- noncomputable def swap : Fin 2 → MvPowerSeries (Fin 2) ℚ :=
--   fun i => X (if i = 0 then 1 else 0)
-- example : (MvPowerSeries.subst swap phi).constantCoeff = 3 := by
--   ...
--
-- (indice : chaque X j est de terme constant nul, donc la substitution ne peut
--  pas creer de terme constant : seul C 3 de phi y contribue. Commencer par
--  montrer constantCoeff (swap i) = 0 pour tout i, puis conclure comme en
--  section 6.)

#eval "Exercice a completer"
─────▶  "Exercice a completer"
--% env 10

Raw input:
{"cmd": "-- Exercice 2 : l'echange des variables preserve le terme constant\n--\n-- TODO etudiant :\n-- noncomputable def swap : Fin 2 \u2192 MvPowerSeries (Fin 2) \u211a :=\n--   fun i => X (if i = 0 then 1 else 0)\n-- example : (MvPowerSeries.subst swap phi).constantCoeff = 3 := by\n--   ...\n--\n-- (indice : chaque X j est de terme constant nul, donc la substitution ne peut\n--  pas creer de terme constant : seul C 3 de phi y contribue. Commencer par\n--  montrer constantCoeff (swap i) = 0 pour tout i, puis conclure comme en\n--  section 6.)\n\n#eval \"Exercice a completer\"", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "\"Exercice a completer\""}],
 "env": 10}

### Exercice 3 : inverser — le critère d'unité

**Objectif** : démontrer le fait d'unité d'une série de terme constant `1` : `IsUnit ((C 1 + X 0 * X 1 : MvPowerSeries (Fin 2) ℚ))` — puis, en une phrase dans le carnet, dire pourquoi ce critère suffit à définir l'inverse d'un élément de groupe formel.

In [12]:
-- Exercice 3 : une serie de terme constant 1 est une unite
--
-- TODO etudiant :
-- example : IsUnit ((C 1 + X 0 * X 1 : MvPowerSeries (Fin 2) ℚ)) := by
--   rw [MvPowerSeries.isUnit_iff_constantCoeff]
--   ...
--
-- (indice : isUnit_iff_constantCoeff reformule le but en l'unite du TERME
--  CONSTANT — une unite de Q, fermable par simp.)

#eval "Exercice a completer"

-- Exercice 3 : une serie de terme constant 1 est une unite
--
-- TODO etudiant :
-- example : IsUnit ((C 1 + X 0 * X 1 : MvPowerSeries (Fin 2) ℚ)) := by
--   rw [MvPowerSeries.isUnit_iff_constantCoeff]
--   ...
--
-- (indice : isUnit_iff_constantCoeff reformule le but en l'unite du TERME
--  CONSTANT — une unite de Q, fermable par simp.)

#eval "Exercice a completer"
─────▶  "Exercice a completer"
--% env 11

Raw input:
{"cmd": "-- Exercice 3 : une serie de terme constant 1 est une unite\n--\n-- TODO etudiant :\n-- example : IsUnit ((C 1 + X 0 * X 1 : MvPowerSeries (Fin 2) \u211a)) := by\n--   rw [MvPowerSeries.isUnit_iff_constantCoeff]\n--   ...\n--\n-- (indice : isUnit_iff_constantCoeff reformule le but en l'unite du TERME\n--  CONSTANT \u2014 une unite de Q, fermable par simp.)\n\n#eval \"Exercice a completer\"", "env": 10}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "\"Exercice a completer\""}],
 "env": 11}

## 11. Conclusion

La marche est posée : une série lue comme fonction-coefficients, les constructeurs `C`/`X`/`monomial`, le produit de Cauchy, l'idéal d'augmentation, la substitution et son garde-fou `HasSubst`, l'inversion par `invOfUnit` avec la série géométrique en témoin, et l'axiome d'identité d'une loi de groupe formel refermé par le lemme de l'organe. Chaque affirmation a été close par le kernel — `simp`, `rw`, `#check` — dans le Mathlib que consomme le lake de Lean-30, sans réimplémentation.

**Prolongements** : [Lean-30](../Lean-30-FormalGroups-Native.ipynb) construit `MvFormalGroup` sur cet échafaudage (morphes, itérés, hauteur) ; [GROTH-01](GROTH-01-Categories-Mathlib-Lean.ipynb) couvre l'autre versant du vocabulaire de la série (catégories, adjonctions, sites) ; la descente des carnets 15/15b/15c viendra occuper les numéros GROTH-02 à 04, et Lean-30 prendra place comme GROTH-06 — la numérotation suit la carte cible de [#17545](https://github.com/jsboige/CoursIA/issues/17545).